# HoVer v6 추론용 Colab 노트북

공개 저장소에 함께 올린 v6 체크포인트를 내려받아, HoVer의 사전 검색된 claim에 대해 **Hop 1·Hop 2 검색 → 중복 Hop 2 제거와 다음 후보 보충 → v6 판정 → 후보와 근거 출력**을 실행합니다. 모델 재학습은 하지 않습니다.

## 실행 전 한 번 설정

1. 공개 GitHub 저장소에 `model/hover_final_verifier_v6_hop2_refill/` 전체를 올립니다. 큰 가중치 파일은 Git LFS를 사용하세요.
2. 아래 **1번 셀**의 `GITHUB_REPO`를 본인 저장소 주소로 바꿉니다.
3. Colab에서 GPU 런타임을 선택한 뒤 위에서부터 셀을 실행합니다.

검색 데이터와 Wikipedia DB는 HoVer 공개 데이터에서 준비합니다. 이 노트북은 저장된 v6 가중치를 불러오므로 학습 셀을 실행하지 않습니다. 지원되는 입력은 HoVer의 TF-IDF 검색 목록에 있는 2-hop claim입니다.


In [ ]:
# 1번: GitHub 저장소에서 공개 v6 가중치를 가져옵니다.
import subprocess
import shutil
from pathlib import Path

GITHUB_REPO = "YOUR_GITHUB_USERNAME/HoVerFactCheck"  # 예: minji/HoVerFactCheck
if GITHUB_REPO.startswith("YOUR_") or "/" not in GITHUB_REPO:
    raise ValueError("GITHUB_REPO를 공개 GitHub의 owner/repository로 바꿔주세요.")

# model.safetensors가 Git LFS에 올라간 경우 실제 파일을 내려받습니다.
if shutil.which("git-lfs") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "git-lfs"], check=True)
subprocess.run(["git", "lfs", "install"], check=True)

REPO_DIR = Path("/content/HoVerFactCheck")
REPO_URL = f"https://github.com/{GITHUB_REPO}.git"
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "lfs", "pull"], check=True)
MODEL_DIR = REPO_DIR / "model" / "hover_final_verifier_v6_hop2_refill"
required_model_files = ["config.json", "model.safetensors", "tokenizer_config.json", "tokenizer.json"]
missing = [name for name in required_model_files if not (MODEL_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"체크포인트 파일을 찾지 못했습니다: {missing} | 경로: {MODEL_DIR}")
print("v6 체크포인트 준비:", MODEL_DIR)
print("체크포인트 파일:", sorted(p.name for p in MODEL_DIR.iterdir() if p.is_file()))


In [ ]:
# 2번: HoVer 공개 데이터와 Wikipedia DB를 준비합니다.
from pathlib import Path
import subprocess

BASE = Path("/content/hover")
if not BASE.exists():
    subprocess.run(["git", "clone", "https://github.com/hover-nlp/hover.git", str(BASE)], check=True)
DB_PATH = BASE / "data/wiki_wo_links.db"
if not DB_PATH.is_file():
    subprocess.run(["bash", "download_data.sh"], cwd=BASE, check=True)
for path in [BASE / "data/hover/hover_train_release_v1.1.json", DB_PATH]:
    if not path.is_file():
        raise FileNotFoundError(f"HoVer 데이터 파일이 없습니다: {path}")
    print("준비됨:", path)


In [ ]:
# 3번: 필요한 NLP 패키지와 spaCy 영어 모델을 준비합니다.
%pip -q install "huggingface_hub>=1.32,<2" sentence-transformers spacy tqdm scikit-learn

import spacy
import sys
if not spacy.util.is_package("en_core_web_sm"):
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)
nlp = spacy.load("en_core_web_sm")
print("spaCy 영어 모델 준비 완료")

## CoreNLP 문장 분리기 준비

In [ ]:
# 4번: CoreNLP 문장 분리 도구를 다운로드하고 압축을 풉니다.
from pathlib import Path
import zipfile
from huggingface_hub import hf_hub_download

CORENLP_DIR = Path("/content/corenlp-unpacked/stanford-corenlp-4.6.0")
if not CORENLP_DIR.exists():
    zip_path = Path(hf_hub_download(
        repo_id="stanfordnlp/CoreNLP",
        filename="stanford-corenlp-latest.zip",
        local_dir="/content/corenlp-download",
    ))
    if not zipfile.is_zipfile(zip_path):
        raise RuntimeError(f"CoreNLP ZIP 파일이 아닙니다: {zip_path}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall("/content/corenlp-unpacked")

print("CoreNLP 폴더:", CORENLP_DIR.exists())
print("JAR 파일 수:", len(list(CORENLP_DIR.glob("*.jar"))))

In [ ]:
# 5번: CoreNLP 문장 분리 서버를 실행하고 연결을 확인합니다.
import json
import subprocess
import time
import requests

def corenlp_is_ready():
    try:
        r = requests.post(
            "http://localhost:9001",
            params={"properties": json.dumps({"annotators": "tokenize,ssplit", "outputFormat": "json"})},
            data="One sentence. Another sentence.",
            timeout=5,
        )
        return r.ok
    except requests.RequestException:
        return False

if not corenlp_is_ready():
    corenlp_log = open("/content/corenlp_server_9001.log", "w")
    corenlp_process = subprocess.Popen(
        ["java", "-mx4g", "-cp", "*",
         "edu.stanford.nlp.pipeline.StanfordCoreNLPServer",
         "-annotators", "tokenize,ssplit", "-port", "9001",
         "-threads", "4", "-timeout", "60000"],
        cwd=str(CORENLP_DIR), stdout=corenlp_log, stderr=corenlp_log,
    )
    for _ in range(30):
        if corenlp_is_ready():
            break
        time.sleep(2)

if not corenlp_is_ready():
    raise RuntimeError("CoreNLP 서버가 9001 포트에서 응답하지 않습니다.")
print("CoreNLP 서버 연결 성공")

In [ ]:
# 6번: HoVer TF-IDF 문서 검색 결과 파일을 준비합니다.
from pathlib import Path
from urllib.request import urlopen
import json
import shutil

retrieval_path = BASE / "data/hover/tfidf_retrieved/train_tfidf_doc_retrieval_results.json"
retrieval_path.parent.mkdir(parents=True, exist_ok=True)
if retrieval_path.is_file():
    with open(retrieval_path, encoding="utf-8") as f:
        retrieval_check = json.load(f)
    print("기존 TF-IDF 검색 결과 사용:", retrieval_path)
else:
    url = "https://nlp.cs.unc.edu/data/hover/train_tfidf_doc_retrieval_results.json"
    temp_path = retrieval_path.with_suffix(".json.part")
    print("TF-IDF 검색 결과 다운로드 중...")
    with urlopen(url, timeout=120) as response, open(temp_path, "wb") as out:
        shutil.copyfileobj(response, out)
    with open(temp_path, encoding="utf-8") as f:
        retrieval_check = json.load(f)
    if not isinstance(retrieval_check, list):
        raise ValueError("검색 결과가 예상한 JSON 리스트 형식이 아닙니다.")
    temp_path.replace(retrieval_path)
print("검색 결과 행 수:", len(retrieval_check))
print(f"파일 크기: {retrieval_path.stat().st_size / 1024**2:.1f} MB")


In [ ]:
# 7번: HoVer의 2-hop claim과 TF-IDF Top-50 문서를 연결합니다.
import json
import sqlite3

train_path = BASE / "data/hover/hover_train_release_v1.1.json"
retrieval_path = BASE / "data/hover/tfidf_retrieved/train_tfidf_doc_retrieval_results.json"
if not retrieval_path.is_file():
    raise FileNotFoundError(f"TF-IDF 검색 결과가 없습니다: {retrieval_path}. 7번 셀을 실행하세요.")
with open(train_path, encoding="utf-8") as f:
    train_raw = json.load(f)
with open(retrieval_path, encoding="utf-8") as f:
    retrieval_raw = json.load(f)
conn = sqlite3.connect(str(DB_PATH), check_same_thread=False)
retrieval_by_claim = {row["claim"]: row for row in retrieval_raw}
two_hop_data = [row for row in train_raw if row["num_hops"] == 2]
titles_by_claim_top50 = {}
for row in two_hop_data:
    retrieval = retrieval_by_claim.get(row["claim"])
    if retrieval:
        titles_by_claim_top50[row["claim"]] = retrieval["doc_retrieval_results"][0][0][:50]
print("사용 가능한 2-hop Claim:", len(titles_by_claim_top50))
if not titles_by_claim_top50:
    raise RuntimeError("HoVer 2-hop Claim과 TF-IDF 목록을 연결하지 못했습니다.")


## 문장 분리와 사전학습 문장 재순위 모델

In [ ]:
# 8번: 문장을 요청 때만 분리합니다. 전체 26만여 문서를 미리 처리하지 않습니다.
import html
import re
import unicodedata
from functools import lru_cache

def normalize_title(title):
    title = unicodedata.normalize("NFKD", str(title))
    title = "".join(ch for ch in title if not unicodedata.combining(ch))
    return " ".join(title.casefold().split())

def clean_sentence(text):
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", html.unescape(text)).strip()

@lru_cache(maxsize=5000)
def get_sentences_corenlp(title):
    title_forms = list(dict.fromkeys([unicodedata.normalize("NFD", title), unicodedata.normalize("NFC", title)]))
    result = None
    for title_form in title_forms:
        result = conn.execute("SELECT text FROM documents WHERE id = ?", (title_form,)).fetchone()
        if result is not None:
            break
    if result is None:
        return ()
    raw_text = result[0]
    response = requests.post(
        "http://localhost:9001",
        params={"properties": json.dumps({"annotators": "tokenize,ssplit", "outputFormat": "json"})},
        data=raw_text.encode("utf-8"),
        headers={"Content-Type": "text/plain; charset=utf-8"},
        timeout=60,
    )
    response.raise_for_status()
    output = []
    for sentence in response.json()["sentences"]:
        tokens = sentence["tokens"]
        output.append(raw_text[tokens[0]["characterOffsetBegin"]:tokens[-1]["characterOffsetEnd"]].strip())
    return tuple(output)

candidate_sentences = {}
def get_title_sentences(title):
    if title not in candidate_sentences:
        candidate_sentences[title] = get_sentences_corenlp(title)
    return candidate_sentences[title]
print("요청 시 문장 분리 준비 완료")


In [ ]:
# 9번: Cross-Encoder 문장 재순위 모델과 점수 함수를 준비합니다.
import numpy as np
import torch
from sentence_transformers import CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
base_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2",
    num_labels=1,
    max_length=256,
    device=device,
)
print("문장 재순위기 로드:", device)

def score_sentence_pairs_fast(pairs, chunk_size=10000, batch_size=128):
    if not pairs:
        return np.array([])
    scores = []
    for start in range(0, len(pairs), chunk_size):
        batch = pairs[start:start + chunk_size]
        batch_scores = base_encoder.predict(
            batch, batch_size=batch_size, show_progress_bar=False
        )
        scores.extend(np.asarray(batch_scores).reshape(-1))
    return np.asarray(scores)

allowed_entity_labels = {"PERSON", "ORG", "GPE", "LOC", "FAC"}

In [ ]:
# 10번: Hop 1·Hop 2 근거 후보 생성 함수를 정의합니다.
import re

def normalize_for_title_match(text):
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9가-힣\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def remove_parenthetical(text):
    return re.sub(r"\s*\([^)]*\)", "", str(text)).strip()

def lookup_entity_titles(entity_name, max_results=20):
    entity_name = str(entity_name).strip()
    if not entity_name:
        return []
    candidates = []
    exact_rows = conn.execute(
        "SELECT id FROM documents WHERE id = ? LIMIT 20", (entity_name,)
    ).fetchall()
    candidates.extend(row[0] for row in exact_rows)

    escaped = entity_name.replace("*", "[*]").replace("?", "[?]")
    prefix_rows = conn.execute(
        "SELECT id FROM documents WHERE id GLOB ? LIMIT ?",
        (escaped + "*", max_results * 3),
    ).fetchall()
    candidates.extend(row[0] for row in prefix_rows)

    unique = list(dict.fromkeys(candidates))
    entity_norm = normalize_for_title_match(entity_name)
    entity_base = normalize_for_title_match(remove_parenthetical(entity_name))
    ranked = []
    for title in unique:
        title_norm = normalize_for_title_match(title)
        title_base = normalize_for_title_match(remove_parenthetical(title))
        if title_norm == entity_norm:
            priority = 0
        elif title_base == entity_base:
            priority = 1
        elif title_norm.startswith(entity_norm):
            priority = 2
        elif entity_norm in title_norm:
            priority = 3
        else:
            priority = 4
        ranked.append((priority, len(title), title))
    ranked.sort(key=lambda x: (x[0], x[1]))
    return [title for _, _, title in ranked[:max_results]]

def build_explicit_two_hop_v5(row, hop1_top_k=5, hop2_top_k=5, max_hop2_titles=100):
    claim = row["claim"]
    if claim not in titles_by_claim_top50:
        raise KeyError("이 claim은 현재 TF-IDF 검색 목록에 없습니다.")

    # Hop 1: Top-50 문서의 전체 문장을 Cross-Encoder로 재순위
    hop1_candidates = []
    for title in titles_by_claim_top50[claim][:50]:
        for sentence_id, sentence in enumerate(get_title_sentences(title)):
            cleaned = clean_sentence(sentence)
            if cleaned:
                hop1_candidates.append({
                    "title": title,
                    "sentence_id": sentence_id,
                    "sentence": cleaned,
                })
    if not hop1_candidates:
        return {"claim": claim, "hop1": [], "entities": [], "hop2": []}

    hop1_scores = score_sentence_pairs_fast([
        (claim, item["sentence"]) for item in hop1_candidates
    ])
    for item, score in zip(hop1_candidates, hop1_scores):
        item["score"] = float(score)
    hop1_ranked = sorted(hop1_candidates, key=lambda x: x["score"], reverse=True)[:hop1_top_k]

    # Claim과 Hop 1 문장에서 연결 개체 추출
    entities = []
    for text in [claim] + [item["sentence"] for item in hop1_ranked]:
        for entity in nlp(text).ents:
            if entity.label_ in allowed_entity_labels:
                name = entity.text.strip()
                if len(name) >= 2 and name not in entities:
                    entities.append(name)

    # Hop 2 문서: 연결 개체의 제목 후보 + Hop 1 문서 + Top-50 fallback
    hop2_titles, seen_titles = [], set()
    def add_title(title):
        key = normalize_title(title)
        if key and key not in seen_titles:
            seen_titles.add(key)
            hop2_titles.append(title)

    for item in hop1_ranked:
        add_title(item["title"])
    for entity_name in entities:
        for title in lookup_entity_titles(entity_name, max_results=20):
            add_title(title)
    if len(hop2_titles) < 20:
        for title in titles_by_claim_top50[claim][:50]:
            add_title(title)
    hop2_titles = hop2_titles[:max_hop2_titles]

    hop2_candidates = []
    seen_facts = set()
    for title in hop2_titles:
        for sentence_id, sentence in enumerate(get_title_sentences(title)):
            fact = (normalize_title(title), sentence_id)
            cleaned = clean_sentence(sentence)
            if fact not in seen_facts and cleaned:
                seen_facts.add(fact)
                hop2_candidates.append({
                    "title": title,
                    "sentence_id": sentence_id,
                    "sentence": cleaned,
                })

    if not hop2_candidates:
        return {"claim": claim, "hop1": hop1_ranked, "entities": entities, "hop2": []}

    hop1_text = "\n".join(item["sentence"] for item in hop1_ranked)
    hop2_query = f"Claim: {claim}\nHop 1 evidence:\n{hop1_text}"
    hop2_scores = score_sentence_pairs_fast([
        (hop2_query, item["sentence"]) for item in hop2_candidates
    ])
    for item, score in zip(hop2_candidates, hop2_scores):
        item["score"] = float(score)
    hop2_ranked = sorted(hop2_candidates, key=lambda x: x["score"], reverse=True)[:hop2_top_k]

    return {"claim": claim, "hop1": hop1_ranked, "entities": entities, "hop2": hop2_ranked}

In [ ]:
# 11번: Hop 2 중복을 제거하고 다음 후보로 채우는 v6 검색 함수를 준비합니다.
import unicodedata
import numpy as np

def _norm_fact_text(text):
    return " ".join(unicodedata.normalize("NFKC", str(text)).casefold().split())

# 12번: Hop 2 중복을 제거하고 다음 후보로 채우는 v6 검색 함수를 준비합니다.
# 모델 학습 때 사용한 v6 Hop 2 보충 검색 로직입니다.
def build_explicit_two_hop_v5_refill(row, hop1_top_k=5, hop2_top_k=5, max_hop2_titles=100):
    claim = row["claim"]
    if claim not in titles_by_claim_top50:
        raise KeyError("이 claim은 현재 TF-IDF 검색 목록에 없습니다.")

    hop1_candidates = []
    for title in titles_by_claim_top50[claim][:50]:
        for sentence_id, sentence in enumerate(get_title_sentences(title)):
            cleaned = clean_sentence(sentence)
            if cleaned:
                hop1_candidates.append({"title": title, "sentence_id": sentence_id, "sentence": cleaned})
    if not hop1_candidates:
        return {"claim": claim, "hop1": [], "entities": [], "hop2": []}

    hop1_scores = score_sentence_pairs_fast([(claim, item["sentence"]) for item in hop1_candidates])
    for item, score in zip(hop1_candidates, hop1_scores):
        item["score"] = float(score)
    hop1_ranked = sorted(hop1_candidates, key=lambda x: x["score"], reverse=True)[:hop1_top_k]

    entities = []
    for text in [claim] + [item["sentence"] for item in hop1_ranked]:
        for entity in nlp(text).ents:
            if entity.label_ in allowed_entity_labels:
                name = entity.text.strip()
                if len(name) >= 2 and name not in entities:
                    entities.append(name)

    hop2_titles, seen_titles = [], set()
    def add_title(title):
        key = normalize_title(title)
        if key and key not in seen_titles:
            seen_titles.add(key)
            hop2_titles.append(title)
    for item in hop1_ranked:
        add_title(item["title"])
    for entity_name in entities:
        for title in lookup_entity_titles(entity_name, max_results=20):
            add_title(title)
    if len(hop2_titles) < 20:
        for title in titles_by_claim_top50[claim][:50]:
            add_title(title)
    hop2_titles = hop2_titles[:max_hop2_titles]

    hop1_fact_keys = {(normalize_title(item["title"]), int(item["sentence_id"])) for item in hop1_ranked}
    hop1_sentence_texts = {_norm_fact_text(item["sentence"]) for item in hop1_ranked}
    hop2_candidates, seen_facts = [], set()
    for title in hop2_titles:
        for sentence_id, sentence in enumerate(get_title_sentences(title)):
            fact = (normalize_title(title), sentence_id)
            cleaned = clean_sentence(sentence)
            if fact in seen_facts or not cleaned:
                continue
            seen_facts.add(fact)
            if fact in hop1_fact_keys or _norm_fact_text(cleaned) in hop1_sentence_texts:
                continue
            hop2_candidates.append({"title": title, "sentence_id": sentence_id, "sentence": cleaned})

    if not hop2_candidates:
        return {"claim": claim, "hop1": hop1_ranked, "entities": entities, "hop2": []}
    hop1_text = "\n".join(item["sentence"] for item in hop1_ranked)
    hop2_query = f"Claim: {claim}\nHop 1 evidence:\n{hop1_text}"
    hop2_scores = score_sentence_pairs_fast([(hop2_query, item["sentence"]) for item in hop2_candidates])
    for item, score in zip(hop2_candidates, hop2_scores):
        item["score"] = float(score)
    hop2_ranked = sorted(hop2_candidates, key=lambda x: x["score"], reverse=True)[:hop2_top_k]
    return {"claim": claim, "hop1": hop1_ranked, "entities": entities, "hop2": hop2_ranked}

build_explicit_two_hop_v6_refill = build_explicit_two_hop_v5_refill
print("v6 Hop 2 보충 검색 함수 준비 완료")


In [ ]:
# 12번: 저장소에 있는 v6 가중치를 불러옵니다. 재학습은 하지 않습니다.
import torch
from sentence_transformers import CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
verifier_v6 = CrossEncoder(str(MODEL_DIR), num_labels=1, max_length=512, device=device)
print("v6 판정 모델 로드:", device)


## Claim 판정

In [ ]:
# 13번: Claim 하나를 골라 label, 확률, 후보 문장, 근거 문장을 확인합니다.
import math
import time

def predict_claim_v6(claim):
    if claim not in titles_by_claim_top50:
        raise KeyError("이 Claim은 HoVer TF-IDF 검색 목록에 없습니다.")
    started = time.perf_counter()
    result = build_explicit_two_hop_v6_refill({"claim": claim})
    candidates, lines = [], []
    for hop in (1, 2):
        for rank, item in enumerate(result.get(f"hop{hop}", [])[:5], 1):
            row = {
                "hop": hop, "rank": rank, "title": item["title"],
                "sentence_id": int(item["sentence_id"]),
                "sentence": item["sentence"], "score": float(item["score"]),
            }
            candidates.append(row)
            lines.append(f"Hop {hop} Evidence {rank} [{row['title']}, 문장 {row['sentence_id']}]: {row['sentence']}")
    evidence_text = "\n".join(lines)
    raw = verifier_v6.predict(
        [[claim, evidence_text]], batch_size=1, show_progress_bar=False,
        activation_fn=torch.nn.Identity(),
    )
    logit = float(np.asarray(raw).reshape(-1)[0])
    probability = 1 / (1 + math.exp(-logit)) if logit >= 0 else math.exp(logit) / (1 + math.exp(logit))
    unique = {}
    for item in candidates:
        key = (unicodedata.normalize("NFD", item["title"]), item["sentence_id"])
        if key not in unique or item["score"] > unique[key]["score"]:
            unique[key] = item
    evidence = sorted(unique.values(), key=lambda x: x["score"], reverse=True)[:2]
    return {
        "claim": claim,
        "label": "SUPPORTED" if probability >= 0.60 else "NOT_SUPPORTED",
        "probability_supported": probability, "threshold": 0.60,
        "latency_ms": (time.perf_counter() - started) * 1000,
        "candidates": candidates, "evidence": evidence,
        "entities": result.get("entities", []),
    }

# 0부터 len(claim_list)-1 사이의 숫자를 바꾸면 다른 사전 검색 HoVer Claim을 선택할 수 있습니다.
claim_list = sorted(titles_by_claim_top50)
claim_index = 0
selected_claim = claim_list[claim_index]
prediction = predict_claim_v6(selected_claim)
print("Claim:", prediction["claim"])
print("판정:", prediction["label"])
print("SUPPORTED 확률:", round(prediction["probability_supported"], 4), "| threshold:", prediction["threshold"])
print("추론 시간(ms):", round(prediction["latency_ms"], 1))
print("후보 문장:")
for item in prediction["candidates"]:
    print(f"- H{item['hop']} #{item['rank']} [{item['title']}] {item['sentence']}")
print("근거 문장:")
for item in prediction["evidence"]:
    print(f"- H{item['hop']} [{item['title']}] {item['sentence']}")
